In [2]:
!pip install monai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.2 MB/s eta 0:00:00


In [5]:
import gc
import itertools
import time
import numpy as np
import torch
import torch.nn.functional as F
import monai
from monai.inferers import sliding_window_inference
from monai.data.utils import compute_importance_map

# 1. Environment
print("=== Environment ===")
print(f"MONAI   : {monai.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM    : "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

print()


# 2. Fast Path implementation

def fast_sliding_window(
    inputs,
    roi_size,
    sw_batch_size,
    predictor,
    overlap=0.25,
    mode="constant",
    sigma_scale=0.125
):
    """
    CUDA-oriented sliding-window inference using:

    - Tensor.unfold() views for patch extraction
    - Z-axis chunking to limit peak memory
    - in-place accumulation with add_()
    - dynamic padding for incomplete windows

    This implementation currently assumes:
    - tensor input/output
    - output spatial size == ROI size
    - constant ou gaussian blending
    """

    device = inputs.device
    batch, channels, depth, height, width = inputs.shape
    roi_d, roi_h, roi_w = roi_size

    stride_d = max(1, int(roi_d * (1.0 - overlap)))
    stride_h = max(1, int(roi_h * (1.0 - overlap)))
    stride_w = max(1, int(roi_w * (1.0 - overlap)))

    # Dynamic padding
    pad_d = (stride_d - ((depth - roi_d) % stride_d)) % stride_d
    pad_h = (stride_h - ((height - roi_h) % stride_h)) % stride_h
    pad_w = (stride_w - ((width - roi_w) % stride_w)) % stride_w

    original_depth = depth
    original_height = height
    original_width = width

    if pad_d or pad_h or pad_w:
        inputs = F.pad(
            inputs,
            (0, pad_w, 0, pad_h, 0, pad_d),
        )

        _, _, depth, height, width = inputs.shape

    if not inputs.is_contiguous():
        inputs = inputs.contiguous()

    # Number of windows in each spatial dimension
    num_d = (depth - roi_d) // stride_d + 1
    num_h = (height - roi_h) // stride_h + 1
    num_w = (width - roi_w) // stride_w + 1

    if mode == "gaussian":
        importance_map = compute_importance_map(
            roi_size, mode="gaussian", sigma_scale=sigma_scale,
            device=device, dtype=inputs.dtype
        )
    else:
        importance_map = None

    # Output buffers
    output = torch.zeros_like(inputs)
    count_map = torch.zeros_like(inputs)

    # Z-axis chunking
    for z_idx in range(num_d):
        d_start = z_idx * stride_d
        # Only keep one ROI-depth slab in the working set.
        slab = inputs[:, :, d_start:d_start + roi_d, :, :]

        # Create views over H/W.
        patches = (
            slab
            .unfold(3, roi_h, stride_h)
            .unfold(4, roi_w, stride_w)
        )

        # Rearrange:
        #   [B, C, D, nH, nW, H, W]
        # -> [B*nH*nW, C, D, H, W]
        patches = (
            patches
            .permute(0, 3, 4, 1, 2, 5, 6)
            .contiguous()
            .view(batch * num_h * num_w, channels, roi_d, roi_h, roi_w)
        )

        # Predictor batches
        for start in range(0, patches.shape[0], sw_batch_size):

            batch_patches = patches[start:start + sw_batch_size]
            predictions = predictor(batch_patches)

            # Stitch predictions immediately.
            for local_idx in range(predictions.shape[0]):

                global_idx = start + local_idx

                batch_idx = global_idx // (num_h * num_w)
                spatial_idx = global_idx % (num_h * num_w)

                h_idx = spatial_idx // num_w
                w_idx = spatial_idx % num_w

                h_start = h_idx * stride_h
                w_start = w_idx * stride_w

                pred_patch = predictions[local_idx]

                if importance_map is not None:
                    # gaussian mode
                    output[
                        batch_idx,
                        :,
                        d_start:d_start + roi_d,
                        h_start:h_start + roi_h,
                        w_start:w_start + roi_w,
                    ].add_(pred_patch * importance_map)

                    count_map[
                        batch_idx,
                        :,
                        d_start:d_start + roi_d,
                        h_start:h_start + roi_h,
                        w_start:w_start + roi_w,
                    ].add_(importance_map)
                else:
                    # constant mode
                    output[
                        batch_idx,
                        :,
                        d_start:d_start + roi_d,
                        h_start:h_start + roi_h,
                        w_start:w_start + roi_w,
                    ].add_(pred_patch)

                    count_map[
                        batch_idx,
                        :,
                        d_start:d_start + roi_d,
                        h_start:h_start + roi_h,
                        w_start:w_start + roi_w,
                    ].add_(1.0)

        # slab / patches go out of scope here

    # Normalize overlapping predictions
    output = output / count_map

    # Remove padding
    output = output[
        :,
        :,
        :original_depth,
        :original_height,
        :original_width,
    ]

    return output


# 3. Predictor used for the microbenchmark

@torch.no_grad()
def identity_predictor(x):
    """
    Deliberately cheap predictor.

    This benchmark isolates the overhead of:
      - patch extraction
      - batching
      - stitching
    """
    return x

# 4. Benchmark utilities

def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def benchmark_function(fn, device, warmup=10, iterations=30):
    """
    Returns:
        mean_ms
        peak_vram_mb
    """

    # Warm-up
    for _ in range(warmup):
        fn()

    synchronize(device)

    # Clean allocator state between benchmark sections.
    if device.type == "cuda":
        torch.cuda.empty_cache()
        synchronize(device)

        torch.cuda.reset_peak_memory_stats(device)

    gc.collect()
    synchronize(device)

    # Timing
    timings = []

    for _ in range(iterations):

        synchronize(device)
        start = time.perf_counter()

        fn()

        synchronize(device)

        timings.append((time.perf_counter() - start) * 1000.0)

    mean_ms = float(np.mean(timings))

    # Peak allocated memory
    if device.type == "cuda":
        peak_bytes = torch.cuda.max_memory_allocated(device)
        peak_vram_mb = peak_bytes / (1024 ** 2)
    else:
        peak_vram_mb = None

    return mean_ms, peak_vram_mb


# 5. Correctness check

def check_correctness(volume, roi_size, sw_batch_size, overlap):

    with torch.no_grad():

        reference = sliding_window_inference(
            volume,
            roi_size,
            sw_batch_size,
            identity_predictor,
            overlap=overlap,
            mode="constant",
        )

        custom = fast_sliding_window(
            volume,
            roi_size,
            sw_batch_size,
            identity_predictor,
            overlap=overlap,
        )

    max_error = (reference - custom).abs().max().item()

    print(
        f"Correctness check | "
        f"overlap={overlap:.2f} | "
        f"shape={tuple(reference.shape)} | "
        f"max error={max_error:.6e}"
    )

    return max_error

# 6. Benchmark configuration
ROI_SIZE = (64, 64, 64)
SW_BATCH_SIZE = 4

WARMUP = 10
ITERATIONS = 30

VOLUMES = [
    (1, 1, 128, 128, 128),
    (1, 1, 160, 160, 160),
]

OVERLAPS = [0.25, 0.50, 0.75]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# 7. Run correctness checks

print("\n=== Correctness ===")

for volume_shape, overlap in itertools.product(VOLUMES, OVERLAPS):

    volume = torch.randn(
        volume_shape,
        device=DEVICE,
        dtype=torch.float32,
    )

    check_correctness(
        volume,
        ROI_SIZE,
        SW_BATCH_SIZE,
        overlap,
    )

    del volume

# 8. Performance benchmark
print("\n=== Performance benchmark ===")

print(
    f"{'Device':<7} | "
    f"{'Volume':<15} | "
    f"{'Overlap':<7} | "
    f"{'MONAI (ms)':>10} | "
    f"{'Fast (ms)':>10} | "
    f"{'Time reduction':>15} | "
    f"{'MONAI peak':>12} | "
    f"{'Fast peak':>12} | "
    f"{'VRAM change':>12}"
)

print("-" * 125)


for volume_shape, overlap in itertools.product(VOLUMES, OVERLAPS):

    volume = torch.randn(
        volume_shape,
        device=DEVICE,
        dtype=torch.float32,
    )

    # MONAI
    def run_monai():
        return sliding_window_inference(
            volume,
            ROI_SIZE,
            SW_BATCH_SIZE,
            identity_predictor,
            overlap=overlap,
            mode="constant",
        )

    monai_ms, monai_vram = benchmark_function(
        run_monai,
        DEVICE,
        warmup=WARMUP,
        iterations=ITERATIONS,
    )

    # Fast Path
    def run_fast():
        return fast_sliding_window(
            volume,
            ROI_SIZE,
            SW_BATCH_SIZE,
            identity_predictor,
            overlap=overlap,
            mode="constant"
        )

    fast_ms, fast_vram = benchmark_function(
        run_fast,
        DEVICE,
        warmup=WARMUP,
        iterations=ITERATIONS,
    )

    # Metrics
    time_reduction = (
        (monai_ms - fast_ms) / monai_ms * 100.0
    )

    if DEVICE.type == "cuda":
        vram_change = (
            (fast_vram - monai_vram)
            / monai_vram
            * 100.0
        )

        monai_vram_str = f"{monai_vram:.0f} MB"
        fast_vram_str = f"{fast_vram:.0f} MB"
        vram_change_str = f"{vram_change:+.1f}%"

    else:
        monai_vram_str = "N/A"
        fast_vram_str = "N/A"
        vram_change_str = "N/A"

    volume_str = "x".join(map(str, volume_shape[2:]))

    print(
        f"{DEVICE.type:<7} | "
        f"{volume_str:<15} | "
        f"{overlap:<7.2f} | "
        f"{monai_ms:>10.2f} | "
        f"{fast_ms:>10.2f} | "
        f"{time_reduction:>14.1f}% | "
        f"{monai_vram_str:>12} | "
        f"{fast_vram_str:>12} | "
        f"{vram_change_str:>12}"
    )

    del volume

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("-" * 125)
print("Benchmark complete.")

=== Environment ===
MONAI   : 1.6.0
PyTorch : 2.11.0+cu128
CUDA    : 12.8
GPU     : Tesla T4
VRAM    : 14.56 GB


=== Correctness ===
Correctness check | overlap=0.25 | shape=(1, 1, 128, 128, 128) | max error=4.768372e-07
Correctness check | overlap=0.50 | shape=(1, 1, 128, 128, 128) | max error=0.000000e+00
Correctness check | overlap=0.75 | shape=(1, 1, 128, 128, 128) | max error=0.000000e+00
Correctness check | overlap=0.25 | shape=(1, 1, 160, 160, 160) | max error=0.000000e+00
Correctness check | overlap=0.50 | shape=(1, 1, 160, 160, 160) | max error=0.000000e+00
Correctness check | overlap=0.75 | shape=(1, 1, 160, 160, 160) | max error=0.000000e+00

=== Performance benchmark ===
Device  | Volume          | Overlap | MONAI (ms) |  Fast (ms) |  Time reduction |   MONAI peak |    Fast peak |  VRAM change
-----------------------------------------------------------------------------------------------------------------------------
cuda    | 128x128x128     | 0.25    |       3.22 |      

In [7]:
# COHORT-SCALE WHOLE-VOLUME INFERENCE
#
# Scenario:
#   A trained 3D segmentation model is applied to a large collection of
#   previously unseen volumes. No training is performed here.
# Why this benchmark?
#   A small per-volume speedup can become relevant when inference is repeated
#   over hundreds or thousands of volumes.


# 1. Lightweight segmentation model
# if it's not lightweight sliding-window cost become marginal
import torch.nn as nn

class Lightweight3DSegmentationModel(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, hidden_channels=4):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv3d(
                in_channels,
                hidden_channels,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(inplace=True),

            nn.Conv3d(
                hidden_channels,
                out_channels,
                kernel_size=1,
            ),
        )

    def forward(self, x):
        return self.network(x)


cohort_model = Lightweight3DSegmentationModel(
    in_channels=1,
    out_channels=1,
    hidden_channels=4,
).to(DEVICE)

cohort_model.eval()


@torch.no_grad()
def cohort_predictor(x):
    return cohort_model(x)

# 2. Cohort configuration
# We use a relatively large 3D volume and a high overlap.
# This deliberately increases the number of windows and therefore the amount
# of sliding-window orchestration.

COHORT_VOLUME_SHAPE = (1, 1, 160, 160, 160)
COHORT_ROI_SIZE = (64, 64, 64)
COHORT_SW_BATCH_SIZE = 4
COHORT_OVERLAP = 0.75

BENCHMARK_CASES = 100

# Number of volumes for the large-scale projection.
# This does NOT mean we actually process this many volumes.
PROJECTED_COHORT_SIZE = 1000

# 3. Generate synthetic cohort
# We keep all volumes on the CPU and move one volume at a time to the
# This is closer to a real inference pipeline than putting the complete cohort
# on the GPU.GPU.

torch.manual_seed(123)

cohort = [
    torch.randn(
        COHORT_VOLUME_SHAPE,
        dtype=torch.float32,
    )
    for _ in range(BENCHMARK_CASES)
]

# 4. Benchmark one complete volume
def benchmark_single_volume(volume, inference_fn):

    volume_gpu = volume.to(DEVICE)

    def run():
        return inference_fn(volume_gpu)

    elapsed_ms, peak_vram = benchmark_function(
        run,
        DEVICE,
        warmup=3,
        iterations=5,
    )

    del volume_gpu

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return elapsed_ms, peak_vram


# 5. Inference functions

def run_monai_cohort(volume):
    return sliding_window_inference(
        inputs=volume,
        roi_size=COHORT_ROI_SIZE,
        sw_batch_size=COHORT_SW_BATCH_SIZE,
        predictor=cohort_predictor,
        overlap=COHORT_OVERLAP,
        mode="constant",
    )


def run_fast_cohort(volume):
    return fast_sliding_window(
        inputs=volume,
        roi_size=COHORT_ROI_SIZE,
        sw_batch_size=COHORT_SW_BATCH_SIZE,
        predictor=cohort_predictor,
        overlap=COHORT_OVERLAP,
    )


# 6. Correctness chec
print("=== Cohort benchmark correctness check ===")
test_volume = cohort[0].to(DEVICE)

with torch.no_grad():

    monai_output = run_monai_cohort(test_volume)

    fast_output = run_fast_cohort(test_volume)

max_error = (monai_output - fast_output).abs().max().item()
mean_error = (monai_output - fast_output).abs().mean().item()

print(f"Max absolute error  : {max_error:.6e}")
print(f"Mean absolute error : {mean_error:.6e}")

del test_volume, monai_output, fast_output

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# 7. Measure each implementation over the cohort
print("\n=== Cohort-scale inference benchmark ===")

monai_times = []
fast_times = []

monai_total = 0.0
fast_total = 0.0

# Warm-up
print("\n=== Warm-up ===")

warmup_volume = cohort[1].to(DEVICE)

with torch.no_grad():
    run_monai_cohort(warmup_volume)
    run_fast_cohort(warmup_volume)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

del warmup_volume


if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

for idx, volume in enumerate(cohort, start=1):
    # Move volume to GPU once, outside the timed regions
    volume_gpu = volume.to(DEVICE)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    # Alternate execution order to reduce systematic timing bias.
    monai_first = (idx % 2 == 1)

    if monai_first:
        # MONAI
        start = time.perf_counter()

        with torch.no_grad():
            run_monai_cohort(volume_gpu)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        monai_elapsed = (time.perf_counter() - start) * 1000.0

        # Fast Path
        start = time.perf_counter()

        with torch.no_grad():
            run_fast_cohort(volume_gpu)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        fast_elapsed = (time.perf_counter() - start) * 1000.0

    else:
        # Fast Path
        start = time.perf_counter()

        with torch.no_grad():
            run_fast_cohort(volume_gpu)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        fast_elapsed = (time.perf_counter() - start) * 1000.0

        # MONAI
        start = time.perf_counter()

        with torch.no_grad():
            run_monai_cohort(volume_gpu)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        monai_elapsed = (time.perf_counter() - start) * 1000.0

    # Store
    monai_times.append(monai_elapsed)
    fast_times.append(fast_elapsed)

    monai_total += monai_elapsed
    fast_total += fast_elapsed

    print(
        f"Case {idx:03d} | "
        f"{'MONAI→Fast' if monai_first else 'Fast→MONAI'} | "
        f"MONAI: {monai_elapsed:8.2f} ms | "
        f"Fast: {fast_elapsed:8.2f} ms | "
        f"Reduction: "
        f"{(monai_elapsed - fast_elapsed) / monai_elapsed * 100:+6.2f}%"
    )

    # Cleanup
    del volume_gpu

# 8. Cohort-level results
average_monai = np.mean(monai_times)
average_fast = np.mean(fast_times)

per_case_saved = average_monai - average_fast

cohort_reduction = (
    (monai_total - fast_total)
    / monai_total
    * 100.0
)

projected_monai_time = (
    average_monai
    * PROJECTED_COHORT_SIZE
)

projected_fast_time = (
    average_fast
    * PROJECTED_COHORT_SIZE
)

projected_saved = (
    projected_monai_time
    - projected_fast_time
)


def format_duration(milliseconds):

    seconds = milliseconds / 1000.0

    if seconds < 60:
        return f"{seconds:.1f} s"

    minutes = seconds / 60.0

    if minutes < 60:
        return f"{minutes:.1f} min"

    hours = minutes / 60.0

    return f"{hours:.2f} h"


print("\n=== Cohort-level results ===")

print(
    f"Average MONAI time / volume : "
    f"{average_monai:.2f} ms"
)

print(
    f"Average Fast time / volume  : "
    f"{average_fast:.2f} ms"
)

print(
    f"Average time saved / volume : "
    f"{per_case_saved:.2f} ms"
)

print(
    f"Overall time reduction      : "
    f"{cohort_reduction:+.2f}%"
)

print("\n=== Projection ===")

print(
    f"Projected cohort size       : "
    f"{PROJECTED_COHORT_SIZE} volumes"
)

print(
    f"MONAI estimated total       : "
    f"{format_duration(projected_monai_time)}"
)

print(
    f"Fast Path estimated total   : "
    f"{format_duration(projected_fast_time)}"
)

print(
    f"Estimated time saved        : "
    f"{format_duration(projected_saved)}"
)

print(
    f"Estimated speedup           : "
    f"{projected_monai_time / projected_fast_time:.2f}x"
)

=== Cohort benchmark correctness check ===
Max absolute error  : 0.000000e+00
Mean absolute error : 0.000000e+00

=== Cohort-scale inference benchmark ===

=== Warm-up ===
Case 001 | MONAI→Fast | MONAI:   183.94 ms | Fast:   178.02 ms | Reduction:  +3.22%
Case 002 | Fast→MONAI | MONAI:   187.37 ms | Fast:   177.02 ms | Reduction:  +5.52%
Case 003 | MONAI→Fast | MONAI:   186.18 ms | Fast:   180.06 ms | Reduction:  +3.29%
Case 004 | Fast→MONAI | MONAI:   185.26 ms | Fast:   176.75 ms | Reduction:  +4.60%
Case 005 | MONAI→Fast | MONAI:   184.22 ms | Fast:   179.82 ms | Reduction:  +2.39%
Case 006 | Fast→MONAI | MONAI:   186.80 ms | Fast:   179.19 ms | Reduction:  +4.07%
Case 007 | MONAI→Fast | MONAI:   184.71 ms | Fast:   181.38 ms | Reduction:  +1.80%
Case 008 | Fast→MONAI | MONAI:   187.45 ms | Fast:   180.91 ms | Reduction:  +3.49%
Case 009 | MONAI→Fast | MONAI:   185.61 ms | Fast:   183.80 ms | Reduction:  +0.97%
Case 010 | Fast→MONAI | MONAI:   189.70 ms | Fast:   180.30 ms | Reducti